# NHANES Mortality × Liver Fibrosis — Project Overview

This notebook summarizes the project structure, data sources, and analysis sequence.
Run the numbered notebooks in order to reproduce all results.

## Research Question

Among US adults enrolled in NHANES (1999–2018), does baseline liver fibrosis — defined
by FIB-4 index ≥ 2.67 — predict excess short-term mortality after accounting for shared
metabolic risk factors (age, sex, BMI, blood pressure, smoking, LDL-C, glucose)?

A companion analysis uses NHIS self-reported liver disease as an alternative exposure.

## Data Sources

| Dataset | Years | N per cycle | Key variables |
|---------|-------|-------------|---------------|
| NHANES continuous | 1999–2018 (10 cycles) | ~5,000–6,500 adults | Labs (AST, ALT, platelets, LDL-C, glucose), exam (BMI, SBP), questionnaire (smoking) |
| NHANES linked mortality (LMF) | Same 10 cycles | All participants | MORTSTAT, UCOD_LEADING (leading cause), PERMTH_EXM |
| NHANES 2017–2018 elastography | 2017–2018 only | ~5,800 adults | LSM (kPa), CAP (dB/m) via FibroScan/VCTE |
| NHIS Sample Adult | 2015–2018 | ~25–33,000/year | Self-reported liver condition (LIVEV), demographics, BMI, smoking, diabetes, hypertension |
| NHIS linked mortality | 2015–2018 | All participants | MORTSTAT, UCOD_LEADING, approximate PERMTH_INT |

**Raw data location:** `data/raw/nhanes/` and `data/raw/lmf/`  
**Derived data:** `data/derived/` (parquet files, generated by notebook 01) and `data/nhis/derived/` (generated by notebook 04)  
**Outputs:** `outputs/tables/` (CSV files) and `outputs/*.md` (summary reports)

## Analysis Sequence

Run notebooks in order. Each notebook saves its outputs for the next.

```
01_data_download.ipynb
  └─ downloads NHANES XPT files (skips if already present)
  └─ downloads LMF .dat files
  └─ merges into per-cycle parquet → data/derived/YYYY_YYYY.parquet

02_analysis.ipynb  [reads: data/derived/]
  └─ FIB-4 and LSM fibrosis classification
  └─ progressive PS matching (3 cohorts: 2007–08, 2011–12, 2017–18)
  └─ KM curves, forest plots, Love plots
  └─ writes: outputs/tables/

03_pooled_analysis.ipynb  [reads: data/derived/]
  └─ pools all 10 cycles (~59,000 adults)
  └─ progressive PS matching on pooled sample
  └─ cause-specific matched KM curves (1999–2014 with full UCOD detail)
  └─ writes: outputs/tables/, outputs/pooled_summary.md

04_nhis_data_download.ipynb
  └─ downloads NHIS Sample Adult CSV files (2015–2018)
  └─ downloads NHIS linked mortality .dat files
  └─ merges and saves → data/nhis/derived/nhis_2015_2018_pooled.parquet

05_nhis_pooled_analysis.ipynb  [reads: data/nhis/derived/]
  └─ replication using self-reported liver condition (LIVEV) as exposure
  └─ progressive PS matching on NHIS pooled sample
  └─ context: why NHIS cannot replicate the FIB-4 analysis

06_fib4_vs_lsm_comparison.ipynb  [reads: data/derived/2017_2018.parquet]
  └─ 2017–2018 only: compares FIB-4 vs two LSM cutpoint schemes
  └─ agreement analysis, progressive matching for each definition
```

## Environment Setup

```bash
cd nhanes_mortality_fibrosis
uv venv
uv pip install -r requirements.txt
# Then launch Jupyter from this directory:
uv run jupyter notebook
```

All notebooks use `os.path.abspath('.')` for paths, so Jupyter must be launched
from the `nhanes_mortality_fibrosis/` directory.

## Key Findings

### Fibrosis exposure definitions

| Definition | Threshold | Fibrosis+ (pooled) | Fibrosis− (pooled) |
|---|---|---|---|
| FIB-4 high | ≥ 2.67 | ~1,821 (~3.1%) | ~37,524 |
| FIB-4 low (reference) | < 1.30 | — | ~37,524 |
| LSM Castera/EASL (2017–18) | ≥ 9.5 kPa (F3/F4) | ~386 (~7.5%) | ~4,274 |
| LSM Eddowes/NAFLD (2017–18) | ≥ 9.7 kPa (F3/F4) | ~370 (~7.5%) | ~4,536 |
| NHIS self-report (2015–18) | LIVEV = 1 | ~1,721 (~1.5%) | ~113,157 |

### Progressive matching: FIB-4, pooled 1999–2018 (24-month window)

| Matching step | RR (95% CI) | N matched pairs |
|---|---|---|
| Crude (unmatched) | 14.1 [11.8–16.7] | 1,821 vs 37,524 |
| + Age, sex | 1.40 [1.07–1.82] | 1,312 pairs |
| + BMI | 1.85 [1.38–2.47] | 1,273 pairs |
| + SBP | 1.62 [1.20–2.18] | 1,222 pairs |
| + Smoking | 1.68 [1.25–2.26] | 1,217 pairs |
| + LDL-C, FPG (fasting subsample) | 2.14 [1.31–3.50] | 570 pairs |

**Interpretation:** The massive crude RR (~14×) reflects age confounding embedded in the
FIB-4 formula. After matching on age and sex, the association drops to ~1.4–1.7×, suggesting
a modest but persistent fibrosis-specific mortality risk beyond shared metabolic risk factors.

### NHIS companion analysis (self-reported liver disease)

| Matching step | RR 24m (95% CI) | RR 36m (95% CI) |
|---|---|---|
| Crude | 3.99 [3.40–4.67] | 3.52 [3.06–4.06] |
| + Age, sex, BMI, smoking, diabetes, hypertension | 2.55 [1.89–3.45] | 1.97 [1.54–2.51] |

Self-reported liver disease predicts elevated mortality consistent with the NHANES FIB-4
finding, but captures only *diagnosed* cases — missing the large pool of undiagnosed NAFLD.

### Limitations

1. **FIB-4 includes age** — crude RR overstates the association; matched estimates are more reliable
2. **Proxy exposures** — FIB-4 and LSM are not histological fibrosis diagnoses
3. **Public-use COD coarsening** — Only 3 UCOD groups available for 2015–2018 cycles
4. **Short follow-up in 2017–2018** — Only ~48% reach 24-month follow-up
5. **Fasting subsample** — LDL-C and FPG restrict Step 5 matching to ~⅓ of participants
6. **No survey weights** — Estimates are not nationally representative

## Output Files

Generated by running notebooks 02–05 (stored in `outputs/`):

| File | Generated by | Contents |
|------|-------------|----------|
| `outputs/tables/cohort_card_by_cohort.csv` | 02 | Sample sizes, FU windows, fibrosis counts per cohort |
| `outputs/tables/mortality_allcause_by_cohort_window_fibrosisdef.csv` | 02 | Death rates by cohort, window, fibrosis definition |
| `outputs/tables/mortality_causegroup_by_cohort_window_fibrosisdef.csv` | 02 | Cause-specific deaths |
| `outputs/tables/progressive_matching_summary.csv` | 02 | Per-cohort progressive matching RRs |
| `outputs/cohort_summary.md` | 02 | Narrative cohort summary |
| `outputs/tables/pooled_descriptive.csv` | 03 | Per-cycle descriptive table (pooled) |
| `outputs/tables/pooled_progressive_matching_24m.csv` | 03 | Pooled progressive matching RRs, 24m |
| `outputs/tables/pooled_progressive_matching_36m.csv` | 03 | Pooled progressive matching RRs, 36m |
| `outputs/pooled_summary.md` | 03 | Pooled analysis narrative summary |

In [ ]:
"""Display key results tables if outputs exist (run notebooks 02-03 first)."""
import os
import pandas as pd

tables_dir = os.path.join(os.path.abspath('.'), 'outputs', 'tables')

pooled_24m = os.path.join(tables_dir, 'pooled_progressive_matching_24m.csv')
if os.path.exists(pooled_24m):
    df = pd.read_csv(pooled_24m)
    print('Pooled progressive matching — 24-month window')
    print(df[['Step', 'N matched pairs', 'RR(24m)', '95%CI(24m)']].to_string(index=False))
else:
    print('outputs/ not found — run notebook 03 to generate results tables.')